# 🏷️ CoreVision — Car **Brand** Classifier Training**Model:** EfficientNetV2-S fine-tuned on VMMRdb  **Task:** Classify car **brand only** (e.g. BMW, Toyota, Ford)  **Expected:** ~50-80 classes, ~85-95% accuracy, ~5-8 min/epoch  This is a **fallback classifier** — when the full make+model classifierisn't confident, we can still identify the brand.## Instructions1. Upload `VMMRdb.zip` to Google Drive at `MyDrive/CoreVision/data/`2. Runtime → GPU (T4)3. Run all cells

In [ ]:
# ============================================================
# CELL 1: Check GPU & Mount Google Drive
# ============================================================
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CoreVision/weights', exist_ok=True)
os.makedirs('/content/drive/MyDrive/CoreVision/data', exist_ok=True)
print('Drive mounted.')

In [ ]:
# ============================================================
# CELL 2: Install Dependencies
# ============================================================
!pip install -q timm
print('Done!')

In [ ]:
# ============================================================
# CELL 3: Extract VMMRdb Dataset
# ============================================================
# Reuses extracted data if model classifier notebook already ran.

import os
import zipfile
import time

LOCAL_DATA = '/content/data'
RAW_DIR = f'{LOCAL_DATA}/_vmmrdb_raw'
FLAT_DIR = f'{LOCAL_DATA}/vmmrdb'
os.makedirs(LOCAL_DATA, exist_ok=True)


def count_images(root):
    count = 0
    if not os.path.exists(root):
        return 0
    for _, _, files in os.walk(root):
        count += sum(1 for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')))
    return count


# Check if data already exists (from model classifier notebook)
if os.path.exists(FLAT_DIR) and count_images(FLAT_DIR) > 100:
    print(f'VMMRdb already extracted: {count_images(FLAT_DIR):,} images in {FLAT_DIR}')
elif os.path.exists(RAW_DIR) and count_images(RAW_DIR) > 100:
    print(f'Raw VMMRdb found: {count_images(RAW_DIR):,} images in {RAW_DIR}')
else:
    # Find and extract ZIP
    drive_data = '/content/drive/MyDrive/CoreVision/data'
    DRIVE_ZIP = None
    if os.path.exists(drive_data):
        for f in os.listdir(drive_data):
            if f.lower().endswith('.zip') and 'vmmr' in f.lower():
                DRIVE_ZIP = os.path.join(drive_data, f)
                break
        if DRIVE_ZIP is None:
            for f in os.listdir(drive_data):
                if f.lower().endswith('.zip'):
                    DRIVE_ZIP = os.path.join(drive_data, f)
                    break

    if DRIVE_ZIP is None:
        raise FileNotFoundError(f'No ZIP found in {drive_data}/')

    print(f'Extracting {os.path.basename(DRIVE_ZIP)}...')
    t0 = time.time()
    os.makedirs(RAW_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(RAW_DIR)
    print(f'Extracted in {time.time() - t0:.0f}s')

print(f'Total images available: {count_images(RAW_DIR if os.path.exists(RAW_DIR) else FLAT_DIR):,}')
print('Cell 3 complete.')

In [ ]:
# ============================================================
# CELL 4: Brand-Level Dataset Preparation
# ============================================================
# VMMRdb dirs look like: acura_cl_1997, bmw_3_series_2015, etc.
# Brand = first word (before first underscore that starts a model name)
# All models and years under a brand get merged.

import os
import re
import random
import shutil
from collections import defaultdict

RAW_DIR = '/content/data/_vmmrdb_raw'
SPLIT_ROOT = '/content/data/brand_split'
MIN_SAMPLES = 50
VAL_RATIO = 0.2
FORCE_REBUILD = True


def is_image(name):
    return name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))


def extract_brand(dir_name):
    """Extract brand from VMMRdb dir name.
    acura_cl_1997 -> acura
    bmw_3_series_2015 -> bmw
    mercedes-benz_c-class_2014 -> mercedes-benz
    land_rover_range_rover_2010 -> land rover
    """
    # Known multi-word brands (lowercase, underscore-separated)
    MULTI_WORD = {
        'aston_martin', 'alfa_romeo', 'am_general',
        'land_rover', 'range_rover',
        'rolls_royce', 'scion_fr',
    }
    name_lower = dir_name.lower()

    # Check multi-word brands first
    for multi in MULTI_WORD:
        if name_lower.startswith(multi + '_'):
            return multi.replace('_', ' ')

    # Default: first word before underscore
    parts = dir_name.split('_')
    return parts[0].lower()


train_dir = f'{SPLIT_ROOT}/train'
val_dir = f'{SPLIT_ROOT}/val'

need_rebuild = FORCE_REBUILD or not os.path.exists(train_dir)

if need_rebuild:
    random.seed(42)
    if os.path.exists(SPLIT_ROOT):
        shutil.rmtree(SPLIT_ROOT)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    # Scan all class directories in raw VMMRdb
    all_dirs = sorted([
        d for d in os.listdir(RAW_DIR)
        if os.path.isdir(os.path.join(RAW_DIR, d))
        and not d.startswith('_')
    ])
    print(f'Found {len(all_dirs)} raw class directories')

    # Group by brand
    brand_images = defaultdict(list)

    for dir_name in all_dirs:
        dir_path = os.path.join(RAW_DIR, dir_name)
        brand = extract_brand(dir_name)

        for f in os.listdir(dir_path):
            fpath = os.path.join(dir_path, f)
            if is_image(f) and os.path.isfile(fpath):
                unique = f'{dir_name}__{f}'
                brand_images[brand].append((fpath, unique))

    print(f'Found {len(brand_images)} unique brands')

    # Show brand distribution
    top = sorted(brand_images.items(), key=lambda x: -len(x[1]))[:25]
    print(f'\nTop 25 brands:')
    for brand, imgs in top:
        print(f'  {brand}: {len(imgs):,} images')

    # Filter + split
    total_train = 0
    total_val = 0
    valid = 0
    skipped = 0

    for brand in sorted(brand_images.keys()):
        images = brand_images[brand]
        if len(images) < MIN_SAMPLES:
            skipped += 1
            continue

        random.shuffle(images)
        split_idx = max(1, min(int(len(images) * (1 - VAL_RATIO)), len(images) - 1))
        train_imgs = images[:split_idx]
        val_imgs = images[split_idx:]

        os.makedirs(os.path.join(train_dir, brand), exist_ok=True)
        os.makedirs(os.path.join(val_dir, brand), exist_ok=True)

        for src, fname in train_imgs:
            shutil.copy2(src, os.path.join(train_dir, brand, fname))
        for src, fname in val_imgs:
            shutil.copy2(src, os.path.join(val_dir, brand, fname))

        total_train += len(train_imgs)
        total_val += len(val_imgs)
        valid += 1

    print(f'\nDataset results:')
    print(f'  Brands found:     {len(brand_images)}')
    print(f'  Skipped (< {MIN_SAMPLES}):  {skipped}')
    print(f'  Valid brands:     {valid}')
    print(f'  Train images:     {total_train:,}')
    print(f'  Val images:       {total_val:,}')
    print(f'  Avg imgs/brand:   {(total_train + total_val) // max(valid, 1)}')
else:
    print('Existing brand split found.')

classes = sorted([c for c in os.listdir(train_dir)
                  if os.path.isdir(os.path.join(train_dir, c))])
n_classes = len(classes)
print(f'\nFinal: {n_classes} brand classes')
print(f'Brands: {classes}')

class_names_path = '/content/drive/MyDrive/CoreVision/data/vmmrdb_brand_classes.txt'
with open(class_names_path, 'w') as f:
    f.write('\n'.join(classes))
print(f'Saved to Drive.')

In [ ]:
# ============================================================
# CELL 5: Hyperparameters
# ============================================================
import os

SPLIT_ROOT = '/content/data/brand_split'

CONFIG = {
    'train_dir': f'{SPLIT_ROOT}/train',
    'val_dir': f'{SPLIT_ROOT}/val',
    'num_classes': n_classes,
    'img_size': 224,
    'batch_size': 256,
    'num_epochs': 15,
    'freeze_epochs': 3,
    'num_workers': 4,
    'lr_backbone': 2e-4,
    'lr_head': 1e-3,
    'weight_decay': 1e-4,
    'save_path': '/content/drive/MyDrive/CoreVision/weights/brand_classifier.pth',
    'resume_from': None,
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'\nPhase 1: Epochs 1-{CONFIG["freeze_epochs"]} (frozen)')
print(f'Phase 2: Epochs {CONFIG["freeze_epochs"]+1}-{CONFIG["num_epochs"]} (fine-tune)')

In [ ]:
# ============================================================
# CELL 6: Data Loaders
# ============================================================
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = CONFIG['img_size']

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(CONFIG['train_dir'], transform=train_transforms)
val_dataset = datasets.ImageFolder(CONFIG['val_dir'], transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=CONFIG['num_workers'],
    pin_memory=True, drop_last=True, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'],
    pin_memory=True, persistent_workers=True
)

print(f'Train: {len(train_dataset)} images, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} images, {len(val_loader)} batches')
print(f'Brands: {len(train_dataset.classes)}')

In [ ]:
# ============================================================
# CELL 7: Build Model
# ============================================================
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


def build_model(num_classes, pretrained=True):
    backbone = timm.create_model('tf_efficientnetv2_s', pretrained=pretrained, num_classes=0)
    feature_dim = backbone.num_features
    model = nn.Sequential(
        backbone,
        nn.Dropout(0.3),
        nn.Linear(feature_dim, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )
    return model, feature_dim


def _get_raw(model):
    if hasattr(model, '_orig_mod'):
        return model._orig_mod
    return model


def freeze_backbone(model):
    for param in _get_raw(model)[0].parameters():
        param.requires_grad = False
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Backbone FROZEN - trainable: {t:,}')


def unfreeze_backbone(model):
    for param in _get_raw(model)[0].parameters():
        param.requires_grad = True
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Backbone UNFROZEN - trainable: {t:,}')


def get_backbone_params(model):
    return list(_get_raw(model)[0].parameters())


def get_head_params(model):
    return list(_get_raw(model)[1:].parameters())


model, feature_dim = build_model(CONFIG['num_classes'])
model = model.to(device)
freeze_backbone(model)

total = sum(p.numel() for p in model.parameters())
print(f'Total params: {total:,}')

In [ ]:
# ============================================================
# CELL 8: Optimizer & Loss
# ============================================================
import torch.optim as optim

optimizer = optim.AdamW(
    [p for p in get_head_params(model) if p.requires_grad],
    lr=CONFIG['lr_head'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['freeze_epochs'], eta_min=1e-5
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print('Optimizer ready (head-only for Phase 1)')

In [ ]:
# ============================================================
# CELL 9: Training Loop
# ============================================================
import time

start_epoch = 0
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
amp_enabled = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

SAVE_DIR = os.path.dirname(CONFIG['save_path'])
LATEST_PATH = os.path.join(SAVE_DIR, 'brand_classifier_latest.pth')


def save_checkpoint(path, epoch, val_acc, val_top5=None, is_best=False):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'accuracy': val_acc,
        'top5_accuracy': val_top5,
        'num_classes': CONFIG['num_classes'],
        'class_names': train_dataset.classes,
        'history': history,
        'best_val_acc': best_val_acc,
        'classifier_type': 'brand',
    }
    torch.save(ckpt, path)
    tag = 'BEST' if is_best else 'latest'
    print(f'  [{tag}] {os.path.basename(path)} (val_acc={val_acc:.2%})')


# Resume
resume_path = CONFIG['resume_from']
if not resume_path or not os.path.exists(str(resume_path or '')):
    if os.path.exists(LATEST_PATH):
        resume_path = LATEST_PATH
    elif os.path.exists(CONFIG['save_path']):
        resume_path = CONFIG['save_path']

if resume_path and os.path.exists(str(resume_path)):
    _ckpt = torch.load(resume_path, map_location='cpu')
    if _ckpt.get('num_classes') == CONFIG['num_classes']:
        model.load_state_dict(_ckpt['model_state_dict'])
        start_epoch = _ckpt['epoch'] + 1
        best_val_acc = _ckpt.get('best_val_acc', 0.0)
        history = _ckpt.get('history', history)
        if start_epoch >= CONFIG['freeze_epochs']:
            unfreeze_backbone(model)
            optimizer = optim.AdamW([
                {'params': get_backbone_params(model), 'lr': CONFIG['lr_backbone']},
                {'params': get_head_params(model), 'lr': CONFIG['lr_head'] * 0.3},
            ], weight_decay=CONFIG['weight_decay'])
            ft = CONFIG['num_epochs'] - CONFIG['freeze_epochs']
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ft, eta_min=1e-6)
        print(f'Resumed from epoch {start_epoch}, best: {best_val_acc:.2%}')
    else:
        print(f'Checkpoint class mismatch. Starting fresh.')
    del _ckpt


def train_epoch(model, loader, optimizer, criterion, scaler, device, ep, total):
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        correct += (out.argmax(1) == labels).sum().item()
        loss_sum += loss.item() * imgs.size(0)
        count += imgs.size(0)
        if i % 50 == 0:
            print(f'  Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}', end='\r')
    return loss_sum / count, correct / count


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, top5_correct, count = 0.0, 0, 0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            out = model(imgs)
            loss = criterion(out, labels)
        correct += (out.argmax(1) == labels).sum().item()
        _, t5 = out.topk(min(5, out.size(1)), dim=1)
        top5_correct += t5.eq(labels.unsqueeze(1)).any(1).sum().item()
        loss_sum += loss.item() * imgs.size(0)
        count += imgs.size(0)
    return loss_sum / count, correct / count, top5_correct / count


FREEZE = CONFIG['freeze_epochs']
TOTAL = CONFIG['num_epochs']
phase2 = start_epoch >= FREEZE

print(f'Training {TOTAL} epochs ({FREEZE} frozen + {TOTAL-FREEZE} fine-tune)')
print('=' * 60)

for epoch in range(start_epoch, TOTAL):
    t0 = time.time()

    if epoch == FREEZE and not phase2:
        phase2 = True
        print('\n' + '=' * 60)
        print('PHASE 2: Unfreezing backbone')
        print('=' * 60)
        unfreeze_backbone(model)
        optimizer = optim.AdamW([
            {'params': get_backbone_params(model), 'lr': CONFIG['lr_backbone']},
            {'params': get_head_params(model), 'lr': CONFIG['lr_head'] * 0.3},
        ], weight_decay=CONFIG['weight_decay'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=TOTAL - FREEZE, eta_min=1e-6)
        scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    tag = 'frozen' if epoch < FREEZE else 'fine-tune'
    print(f'\nEpoch {epoch+1}/{TOTAL} [{tag}]')

    tl, ta = train_epoch(model, train_loader, optimizer, criterion, scaler, device, epoch, TOTAL)
    vl, va, v5 = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(tl)
    history['train_acc'].append(ta)
    history['val_loss'].append(vl)
    history['val_acc'].append(va)

    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:02d}/{TOTAL} [{tag}] '
          f'| Train: {tl:.4f} {ta:.2%} '
          f'| Val: {vl:.4f} {va:.2%} Top5: {v5:.2%} '
          f'| {elapsed:.0f}s')

    if va > best_val_acc:
        best_val_acc = va
        save_checkpoint(CONFIG['save_path'], epoch, va, v5, is_best=True)
    save_checkpoint(LATEST_PATH, epoch, va, v5, is_best=False)

print(f'\nDone! Best val accuracy: {best_val_acc:.2%}')

In [ ]:
# ============================================================
# CELL 10: Plot Training History
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)
freeze_ep = CONFIG.get('freeze_epochs', 3)

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', ms=3)
axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val', ms=3)
axes[0].axvline(x=freeze_ep + 0.5, color='green', ls='--', alpha=0.7, label='Unfreeze')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train', ms=3)
axes[1].plot(epochs, [a*100 for a in history['val_acc']], 'r-o', label='Val', ms=3)
axes[1].axvline(x=freeze_ep + 0.5, color='green', ls='--', alpha=0.7, label='Unfreeze')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

plt.suptitle(f'Brand Classifier (VMMRdb) | Best: {best_val_acc:.2%}', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CoreVision/brand_training_history.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 11: Save & Download
# ============================================================
print('Files in Drive:')
!ls -lh "/content/drive/MyDrive/CoreVision/weights/"
print()
print('Download these to your project:')
print('  brand_classifier.pth       -> weights/brand_classifier.pth')
print('  vmmrdb_brand_classes.txt   -> data/vmmrdb_brand_classes.txt')